In [24]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
from utils import download_and_save

In [3]:
df = download_and_save("https://raw.githubusercontent.com/hovhannisyan91/aca/refs/heads/main/lab/python/data/cohort/cohort_analysis.csv", "../data/cohort/cohort_analysis.csv")

In [4]:
df.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
0,1,2024-07-01,2025-02-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard
1,2,2024-04-01,2024-05-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard
2,3,2024-05-01,2024-07-01,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard
3,4,2024-07-01,NaN,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic


In [5]:
print(df.shape)
print(df.columns)
print(df.info())

(12000, 12)
Index(['user_id', 'acquisition_date', 'cancellation_month', 'gender',
       'marital_status', 'age', 'income_segment', 'country', 'channel',
       'campaign_id', 'device_type', 'plan_type'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   user_id             12000 non-null  int64
 1   acquisition_date    12000 non-null  str  
 2   cancellation_month  7066 non-null   str  
 3   gender              12000 non-null  str  
 4   marital_status      12000 non-null  str  
 5   age                 12000 non-null  int64
 6   income_segment      11369 non-null  str  
 7   country             12000 non-null  str  
 8   channel             12000 non-null  str  
 9   campaign_id         12000 non-null  str  
 10  device_type         12000 non-null  str  
 11  plan_type           12000 non-null  str  
dtypes: int64(2), st

In [6]:
df.isna().sum()

user_id                  0
acquisition_date         0
cancellation_month    4934
gender                   0
marital_status           0
age                      0
income_segment         631
country                  0
channel                  0
campaign_id              0
device_type              0
plan_type                0
dtype: int64

In [7]:
df.describe(include="all")

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
count,12000.00000,12000,7066,12000,12000,12000.000000,11369,12000,12000,12000,12000,12000
unique,NaN,8,19,2,2,NaN,4,12,3,9,3,3
top,NaN,2024-05-01,2024-05-01,Female,Single,NaN,High,Netherlands,Paid Ads,Paid Ads_A,Android,Standard
freq,NaN,1541,932,6016,7087,NaN,4823,1057,4854,1636,6582,5979
mean,6000.50000,NaN,NaN,NaN,NaN,34.630250,NaN,NaN,NaN,NaN,NaN,NaN
std,3464.24595,NaN,NaN,NaN,NaN,9.514896,NaN,NaN,NaN,NaN,NaN,NaN
min,1.00000,NaN,NaN,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,3000.75000,NaN,NaN,NaN,NaN,28.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,6000.50000,NaN,NaN,NaN,NaN,34.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,9000.25000,NaN,NaN,NaN,NaN,41.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df["acquisition_month"] = pd.to_datetime(df["acquisition_date"]).dt.to_period("M")

df[["user_id", "acquisition_date", "acquisition_month"]].head()

,user_id,acquisition_date,acquisition_month
0,1,2024-07-01,2024-07
1,2,2024-04-01,2024-04
2,3,2024-05-01,2024-05
3,4,2024-07-01,2024-07
4,5,2024-03-01,2024-03


In [9]:
df["is_churned"] = df["cancellation_month"].notna().astype(int)
df["is_churned"].value_counts()

is_churned
1    7066
0    4934
Name: count, dtype: int64

In [10]:
df["acquisition_date"] = pd.to_datetime(df["acquisition_date"], errors="coerce")
df["cancellation_month"] = pd.to_datetime(df["cancellation_month"], errors="coerce")

df["month_churned"] = np.where(
    df["cancellation_month"].notna(),
    (
        (df["cancellation_month"].dt.year - df["acquisition_date"].dt.year) * 12
        + (df["cancellation_month"].dt.month - df["acquisition_date"].dt.month)
    ),
    np.nan
)

df[["user_id", "acquisition_date", "acquisition_month","cancellation_month", "month_churned"]].head(10)

,user_id,acquisition_date,acquisition_month,cancellation_month,month_churned
0,1,2024-07-01,2024-07,2025-02-01,7.0
1,2,2024-04-01,2024-04,2024-05-01,1.0
2,3,2024-05-01,2024-05,2024-07-01,2.0
3,4,2024-07-01,2024-07,NaT,NaN
4,5,2024-03-01,2024-03,2024-04-01,1.0
5,6,2024-08-01,2024-08,NaT,NaN
6,7,2024-05-01,2024-05,NaT,NaN
7,8,2024-05-01,2024-05,2024-06-01,1.0
8,9,2024-07-01,2024-07,NaT,NaN
9,10,2024-02-01,2024-02,2024-12-01,10.0


#### Data Preperation | Keep Only Valid Churn Months (12)

In [11]:
df["month_churned"] = df["month_churned"].where(
    df["month_churned"].between(1, 12),
    np.nan
)

df["month_churned"].value_counts(dropna=False).sort_index()

month_churned
1.0     2392
2.0     1363
3.0      845
4.0      585
5.0      443
6.0      362
7.0      266
8.0      248
9.0      182
10.0     168
11.0     113
12.0      99
NaN     4934
Name: count, dtype: int64

#### Data Preperation | Cohort Size

In [12]:
cohort_size = (
    df.groupby("acquisition_month")
      .agg(new_users=("user_id", "count"))
      .reset_index()
)

cohort_size

,acquisition_month,new_users
0,2024-01,1502
1,2024-02,1528
2,2024-03,1465
3,2024-04,1509
4,2024-05,1541
5,2024-06,1493
6,2024-07,1500
7,2024-08,1462


#### Data Preperation | Count Actual Churn Events by Cohort and Tenure Month

In [13]:
cohort_churn = (
    df[df["month_churned"].notna()]
    .groupby(["acquisition_month", "month_churned"])
    .agg(users=("user_id", "count"))
    .reset_index()
)

cohort_churn["month_churned"] = cohort_churn["month_churned"].astype(int)

cohort_churn.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,148
1,2024-01,2,95
2,2024-01,3,79
3,2024-01,4,76
4,2024-01,5,52
5,2024-01,6,49
6,2024-01,7,41
7,2024-01,8,37
8,2024-01,9,24
9,2024-01,10,26


#### Data Preperation | Full 12-Month Grid

In [14]:
max_month = 12
tenure_range = np.arange(1, max_month + 1)

full_index = pd.MultiIndex.from_product(
    [cohort_size["acquisition_month"].unique(), tenure_range],
    names=["acquisition_month", "month_churned"]
)

cohort_data = (
    cohort_churn
    .set_index(["acquisition_month", "month_churned"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

cohort_data.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,148
1,2024-01,2,95
2,2024-01,3,79
3,2024-01,4,76
4,2024-01,5,52
5,2024-01,6,49
6,2024-01,7,41
7,2024-01,8,37
8,2024-01,9,24
9,2024-01,10,26


#### Data Preperation | Merge Cohort Size

In [15]:
cohort_data = cohort_data.merge(cohort_size, on="acquisition_month", how="left")
cohort_data.head(15)

,acquisition_month,month_churned,users,new_users
0,2024-01,1,148,1502
1,2024-01,2,95,1502
2,2024-01,3,79,1502
3,2024-01,4,76,1502
4,2024-01,5,52,1502
5,2024-01,6,49,1502
6,2024-01,7,41,1502
7,2024-01,8,37,1502
8,2024-01,9,24,1502
9,2024-01,10,26,1502


#### Data Preperation | Calculate Metrics

In [18]:
cohort_data = cohort_data.sort_values(["acquisition_month", "month_churned"])

cohort_data["cumulative_churn"] = (
    cohort_data.groupby("acquisition_month")["users"].cumsum()
)

cohort_data["active_users"] = (
    cohort_data["new_users"] - cohort_data["cumulative_churn"]
)

cohort_data["retention_rate"] = (
    cohort_data["active_users"] / cohort_data["new_users"]
)

cohort_data["acquisition_month"] = cohort_data["acquisition_month"].astype(str)

cohort_data.head(15)

,acquisition_month,month_churned,users,new_users,cumulative_churn,active_users,retention_rate
0,2024-01,1,148,1502,148,1354,0.901465
1,2024-01,2,95,1502,243,1259,0.838216
2,2024-01,3,79,1502,322,1180,0.785619
3,2024-01,4,76,1502,398,1104,0.735020
4,2024-01,5,52,1502,450,1052,0.700399
5,2024-01,6,49,1502,499,1003,0.667776
6,2024-01,7,41,1502,540,962,0.640479
7,2024-01,8,37,1502,577,925,0.615846
8,2024-01,9,24,1502,601,901,0.599867
9,2024-01,10,26,1502,627,875,0.582557


## Retention by Cohort

### Retention Line Plots | Gender Level

In [19]:
gender_data = (
    df[df["month_churned"].notna()]
    .groupby(["gender", "month_churned"])
    .agg(churned_users=("user_id", "count"))
    .reset_index()
)
gender_data.head()

,gender,month_churned,churned_users
0,Female,1.0,1200
1,Female,2.0,702
2,Female,3.0,409
3,Female,4.0,294
4,Female,5.0,227


In [ ]:
gender_base = (
    df.groupby("gender")
    .agg(total_users=("user_id", "count"))
    .reset_index()
)

gender_base.head()

,gender,total_users
0,Female,6016
1,Male,5984


In [22]:
gender_data = gender_data.merge(
    gender_base,
    on="gender",
    how="left"
)

gender_data["churn_rate"] = (
    gender_data["churned_users"] / gender_data["total_users"]
)

gender_data.head()

,gender,month_churned,churned_users,total_users,churn_rate
0,Female,1.0,1200,6016,0.199468
1,Female,2.0,702,6016,0.116689
2,Female,3.0,409,6016,0.067985
3,Female,4.0,294,6016,0.048870
4,Female,5.0,227,6016,0.037733


In [23]:
fig = px.line(
    gender_data,
    x="month_churned",
    y="churn_rate",
    color="gender",
    markers=True,
    title="Monthly Churn Rate by Gender",
    labels={
        "month_churned": "Month Since Acquisition",
        "churn_rate": "Churn Rate",
        "gender": "Gender"
    }
)

fig.update_layout(
    yaxis_tickformat=".0%",
    xaxis=dict(dtick=1),
    hovermode="x unified"
)

fig.show()

### Retention Line Plots | Marital Status Level

In [27]:
marital_status_data = (
    df[df["month_churned"].notna()]
    .groupby(["marital_status", "month_churned"])
    .agg(churned_users=("user_id", "count"))
    .reset_index()
)
marital_status_data.head()

,marital_status,month_churned,churned_users
0,Married,1.0,979
1,Married,2.0,570
2,Married,3.0,359
3,Married,4.0,231
4,Married,5.0,177


In [28]:
marital_status_base = (
    df.groupby("marital_status")
    .agg(total_users=("user_id", "count"))
    .reset_index()
)

marital_status_base.head()

,marital_status,total_users
0,Married,4913
1,Single,7087


In [29]:
marital_status_data = marital_status_data.merge(
    marital_status_base,
    on="marital_status",
    how="left"
)

marital_status_data["churn_rate"] = (
    marital_status_data["churned_users"] / marital_status_data["total_users"]
)

marital_status_data.head()

,marital_status,month_churned,churned_users,total_users,churn_rate
0,Married,1.0,979,4913,0.199267
1,Married,2.0,570,4913,0.116019
2,Married,3.0,359,4913,0.073071
3,Married,4.0,231,4913,0.047018
4,Married,5.0,177,4913,0.036027


In [30]:
fig = px.line(
    marital_status_data,
    x="month_churned",
    y="churn_rate",
    color="marital_status",
    markers=True,
    title="Monthly Churn Rate by Marital Status",
    labels={
        "month_churned": "Month Since Acquisition",
        "churn_rate": "Churn Rate",
        "marital_status": "Marital Status"
    }
)

fig.update_layout(
    yaxis_tickformat=".0%",
    xaxis=dict(dtick=1),
    hovermode="x unified"
)

fig.show()